# Persistence and Streaming

Agents often work on longer running tasks.

For these task, agents need be built on these two concepts: 

1. Persistence - It helps to keep around the state of an agent at a point in time. This lets you go back and resume in the previous state. It is very important for long running applications.
2. Streaming - Emit list of signals of whats happening in the agent at a moment

#### _Create Agent:_

Import langraph and langchain tools:

    from langgraph.graph import StateGraph, END
    from typing import TypedDict, Annotated
    import operator
    from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage
    from langchain_openai import ChatOpenAI
    from langchain_community.tools.tavily_search import TavilySearchResults

Create tavily search tool:

    tool = TavilySearchResults(max_results=2)

Create Agent state of typed dictionary with messages and operator to add messages:

    class AgentState(TypedDict):
        messages: Annotated[list[AnyMessage], operator.add]

Create Agent using LangGraph:

#### _Persistence:_

Add persistence to the agent :-
- Use a check pointer to be able to access any state
- A checkpoint is used to checkpoint the state after and between every node
- Add it using SqliteSaver for memory which a simple checkpoint that uses sqlite i.e a built in database under the hood. Use in memory database for this example.
- Add another parameter to agent called checkpointer and then pass it to the grpah compile to modify the agent

    from langgraph.checkpoint.sqlite import SqliteSaver

    memory = SqliteSaver.from_conn_string(":memory:")

    class Agent:
        def __init__(self, model, tools, checkpointer, system=""):
            self.system = system
            graph = StateGraph(AgentState)
            graph.add_node("llm", self.call_openai)
            graph.add_node("action", self.take_action)
            graph.add_conditional_edges("llm", self.exists_action, {True: "action", False: END})
            graph.add_edge("action", "llm")
            graph.set_entry_point("llm")
            self.graph = graph.compile(checkpointer=checkpointer)
            self.tools = {t.name: t for t in tools}
            self.model = model.bind_tools(tools)

        def call_openai(self, state: AgentState):
            messages = state['messages']
            if self.system:
                messages = [SystemMessage(content=self.system)] + messages
            message = self.model.invoke(messages)
            return {'messages': [message]}

        def exists_action(self, state: AgentState):
            result = state['messages'][-1]
            return len(result.tool_calls) > 0

        def take_action(self, state: AgentState):
            tool_calls = state['messages'][-1].tool_calls
            results = []
            for t in tool_calls:
                print(f"Calling: {t}")
                result = self.tools[t['name']].invoke(t['args'])
                results.append(ToolMessage(tool_call_id=t['id'], name=t['name'], content=str(result)))
            print("Back to the model!")
            return {'messages': results}

Initialise agent with model, tools, system prompt and memory checkpointer:

    prompt = """You are a smart research assistant. Use the search engine to look up information. \
    You are allowed to make multiple calls (either together or in sequence). \
    Only look up information when you are sure of what you want. \
    If you need to look up some information before asking a follow up question, you are allowed to do that!
    """
    model = ChatOpenAI(model="gpt-4o")
    abot = Agent(model, [tool], system=prompt, checkpointer=memory)

#### _Streaming:_

- Stream helps view all intermediate results within the agent
- It gives good visibility into what is happening
- Can stream the individual messages which could be the AI message that determines the action to take and then the observation which is the result after completing the action
- Can stream tokens where for each token of the LLM call, we might stream the output.


Create human message:

    messages = [HumanMessage(content="What is the weather in sf?")]

Add Thread config :-
- This is used to keep track of different threads inside the persistence checkpointer
- This allows to have multiple conversations at the same time
- Useful for production applications to manage several users

    thread = {"configurable": {"thread_id": "1"}}

Call graph with Stream and pass in the messages dictionary and the thread config:

=> This will return a stream of events which represent updates with the state over time 

=> Since each state has only one key, we can loop through and print it out

    for event in abot.graph.stream({"messages": messages}, thread):
        for v in event.values():
            print(v['messages'])

The Streaming Results from agent which shows what is happenning:

=> First, it returns an AI message which is the first result from the LLM to call tavily search tool

=> Then, it returns a Tool message which is the result of calling tavily search 

=> In the end, it returns an AI message with the final answer from the LLM for the query

##### EXAMPLE 2:

Create another human message query:

    messages = [HumanMessage(content="What about in la?")]

Configure thread and stream the graph:

=> This example is a follow up question as it continues in the same conversation with the previous message 

=> Since it is a follow up question, the agent should expect the query to be about the weather even when it is not explicitly specified

=> To make sure the messages follows the same conversation, use the same config thread as before with the thread id as 1

    thread = {"configurable": {"thread_id": "1"}}
    for event in abot.graph.stream({"messages": messages}, thread):
        for v in event.values():
            print(v)

=> First, it returns an AI message which is the first result from the LLM to call tavily search tool to get the weather for LA even thought weather wasn't explicitly mentioned in the follow-up query

=> Then, it returns a Tool message which is the result of calling tavily search 

=> In the end, it returns an AI message with the final answer from the LLM for the query

#### _Stream Tokens:_

- Can stream tokens using A-stream events method
- A-stream event is an asynchronous method which means we should use an async checkpointer
- Use async sqlite saver as the agent checkpoinnter
- It looks for events that correspond to new tokens 
- Stream out the tokens one at a time with the pipe line delimiter

Import Async Sqlite saver and use it as the agent checkpointer:

    from langgraph.checkpoint.aiosqlite import AsyncSqliteSaver
    memory = AsyncSqliteSaver.from_conn_string(":memory:")

Intialise agent with model, tool, system prompt, async sqlite saver memory as checkpointer:

    abot = Agent(model, [tool], system=prompt, checkpointer=memory)

Create new human message and config thread as new conversation with new id:

    messages = [HumanMessage(content="What is the weather in SF?")]
    thread = {"configurable": {"thread_id": "4"}}

Call agent with astream event and pass on the messages dictionary, thread and version:

    async for event in abot.graph.astream_events({"messages": messages}, thread, version="v1"):
        kind = event["event"]
        if kind == "on_chat_model_stream":
            content = event["data"]["chunk"].content
            if content:
                # Empty content in the context of OpenAI means
                # that the model is asking for a tool to be invoked.
                # So we only print non-empty content
                print(content, end="|")

=> If event is an on chat model stream as it corresponds to new tokens, then we print non-empty content from the model